In [1]:
# install beautifulsoup for the first time if not there
#!conda install BeautifulSoup4
#!pip install BeautifulSoup4
#!pip install html5lib

In [1]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
from IPython.display import display, HTML

In [2]:
html="""<html>
    <head>
        <title>
            This is an example page
        </title>
    </head>
    <body>
        Here is the content of the page
        <p>
            One paragraph
        </p>
        <li> one </li>
        <li> two </li>
        <table>
            <tr>
                <td>col 1</td>
                <td>col 2</td>
            </tr>
            <tr>
                <td>val 1</td>
                <td>val 2</td>
            </tr>
        </table>
        End of body
    </body>
</html>"""

In [3]:
HTML(html)

col 1,col 2
val 1,val 2


In [4]:
root = ET.fromstring(html)

In [5]:
for element in root.iter('td'):
    print(element.text)

col 1
col 2
val 1
val 2


In [6]:
messed_html = html + "blablabla"
root = ET.fromstring(messed_html)

ParseError: junk after document element: line 26, column 7 (<string>)

In [7]:
soup = BeautifulSoup(messed_html, 'html')

In [8]:
for element in soup.find_all('td'):
    print(element.text)

col 1
col 2
val 1
val 2


In [9]:
def print_html_tree(tag,level = 0):
    print('\t'*level + tag.name)
    for child in tag.findChildren():
        print_html_tree(child, level+1)

In [10]:
print_html_tree(soup.currentTag)

[document]
	html
		head
			title
		title
		body
			p
			li
			li
			table
				tr
					td
					td
				td
				td
				tr
					td
					td
				td
				td
			tr
				td
				td
			td
			td
			tr
				td
				td
			td
			td
		p
		li
		li
		table
			tr
				td
				td
			td
			td
			tr
				td
				td
			td
			td
		tr
			td
			td
		td
		td
		tr
			td
			td
		td
		td
	head
		title
	title
	body
		p
		li
		li
		table
			tr
				td
				td
			td
			td
			tr
				td
				td
			td
			td
		tr
			td
			td
		td
		td
		tr
			td
			td
		td
		td
	p
	li
	li
	table
		tr
			td
			td
		td
		td
		tr
			td
			td
		td
		td
	tr
		td
		td
	td
	td
	tr
		td
		td
	td
	td
	html


/tmp/ipykernel_68/2094678356.py:3: DeprecationWarning: Call to deprecated method findChildren. (Replaced by find_all) -- Deprecated since version 3.0.0.
  for child in tag.findChildren():


Extract names and titles of [MIS Faulty at UGA](https://www.terry.uga.edu/directory/?group=management-information-systems)

How many assistant professors are there?

In [14]:
r = requests.get("https://www.terry.uga.edu/directory/?group=management-information-systems")

In [15]:
r.status_code # nolonger works

403

In [15]:
### html = r.text

In [19]:
with open("data/mis.html", encoding="utf-8") as f:
    html = '\n'.join(f.readlines())


In [20]:
print(html)

<!doctype html>

<html lang="en-US">

<head>

	<meta charset="UTF-8">

	<meta name="viewport" content="width=device-width, initial-scale=1">

	<link rel="profile" href="https://gmpg.org/xfn/11">



	

        <script type="text/javascript" data-pressidium-cc-no-block>

            function onPressidiumCookieConsentUpdated(event) {

                window.dataLayer = window.dataLayer || [];

                window.dataLayer.push({

                    event: event.type,

                    consent: event.detail,

                });

            }



            window.addEventListener('pressidium-cookie-consent-accepted', onPressidiumCookieConsentUpdated);

            window.addEventListener('pressidium-cookie-consent-changed', onPressidiumCookieConsentUpdated);

        </script>



        

        <style id="pressidium-cc-styles">

            .pressidium-cc-theme {

                --cc-bg: #fffafa;

--cc-text: #55111a;

--cc-btn-primary-bg: #ba0c2f;

--cc-btn-primary-text: #fff

In [21]:
HTML(html)

In [22]:
root = ET.fromstring(html)

ParseError: syntax error: line 1, column 0 (<string>)

In [23]:
soup = BeautifulSoup(html, 'html')

In [24]:
names = soup.find_all('a', attrs={'class':'entry-title'}, )
names = [name for name in names if name.text != ""]
urls = [name.attrs['href'] for name in names]
names = [name.text for name in names]
len(names)

58

In [25]:
positions = soup.find_all('span', attrs={'class':'position__title'})
len(positions)

62

In [26]:
# lengths of names and postisions do not match, need to find another way

In [27]:
cards = soup.find_all('div', attrs={'class':'col-12 col-sm-8'})
len(cards)

58

In [28]:
people = []
for card in cards:
    name=card.find('a', attrs={'class':                          
                         'entry-title'}).text
    position=card.find('span', attrs={'class':
                         'position__title'}).text
    person = {'name':name, 'position':position}
    people.append(person)
#people
df = pd.DataFrame(people)

In [29]:
df.head()

,name,position
0,Karen Aguar,Senior Lecturer
1,John-Patrick (JP) Akinyemi,Assistant Professor
2,Marie-Claude (Maric) Boudreau,Brown Family Professor of Business Sustainabil...
3,Eunsol Cho,Assistant Professor
4,Chris Colson,Part-Time Lecturer


In [30]:
sum(df.position == 'Assistant Professor')

7

## using pandas read_html

The web page has to have a table

In [31]:
response = requests.get("https://sicemdawgs.com/uga-football-roster/")
response.status_code

200

In [35]:
from io import StringIO

In [36]:
dfs = pd.read_html(StringIO(response.text))

In [37]:
type(dfs)

list

In [38]:
len(dfs)

1

In [39]:
df1 = dfs[0]
df1

,No.,Name,Pos.,Cl.,Ht.,Wt.,Hometown (High School)
0,0,Gabe Harris Jr.,OLB,Sr.,6-4,260,"Thomasville, GA (IMG Academy)"
1,0,Sacovie White-Helton,WR,Jr.,5-9,180,"Cartersville, GA (Cass)"
2,1,Ellis Robinson IV,DB,Jr.,6-0,180,"New Haven, CT (IMG Academy)"
3,1,Talyn Taylor,WR,So.,6-0,190,"Geneva, IL (Geneva)"
4,2,Thomas Blackshear,WR,So.,6-1,200,"Savannah, GA (Calvary Day School)"
...,...,...,...,...,...,...,...
123,96,JJ Hanne,DL,So.,6-4,290,"Berlin, Germany (Santa Margarita Cath.)"
124,97,Wade Register,P,Fr.,6-3,225,"Dublin, GA (Trinity Christian)"
125,98,Connor Ferguson,PK,So.,5-9,155,"Savannah, GA (Benedictine Military)"
126,99,Joseph Jonah-Ajonye,DL,Jr.,6-4,275,"Shenandoah, TX (Oak Ridge)"
